# CRIVO — Pointer/Copy Causal v3

**Pergunta:** com o desenho experimental corrigido, a mesma cabeça pointer/copy do v2 preserva nomes arbitrários e para no lugar certo, sem alterar nenhum peso do Transformer?

O que mudou em relação ao v2 (resultado do v2 preservado em `experimentos/pointer_copy_causal_v2_20261012/RESULTADO.md` e no Drive em `experimento_pointer_copy/pointer_copy_20261012_042442/`, sem sobrescrita):

- a mensagem é escrita como **texto completo** e tokenizada por `lp.fonte_dialogo`; a resposta por `lp.codificar_texto(texto) + <fim>` — exatamente o runtime do chat;
- **nenhum ID de nome é colado** separadamente: no meio da frase o nome começa com `Ġ`, no início da resposta não;
- o **`<fim>` da cópia pura é supervisionado explicitamente** (termo `0.50 * fim_copia_ce`); acertar a cópia = nome inteiro **e** parar;
- **blind de regressão:** as mesmas 100 entidades do blind v2, reproduzidas pelo código do v2, avaliadas no formato antigo (comparação direta) e no formato real;
- **blind novo:** 100 entidades inéditas (disjuntas de tudo que o v2 usou), 600 casos com a tokenização do runtime, **congelados no repositório antes deste treino** e conferidos por SHA-256;
- mesma base (`c0a56b15…`, fingerprint `257a4eb4…`), Transformer e `lm_head` congelados;
- **mesma arquitetura do pointer** (código idêntico ao v2), **reinicializada** — o `pointer_head.pt` do v2 não é carregado;
- mesmos hiperparâmetros do v2; métricas separadas por tarefa: exact-name, exact-name+fim, categoria, função, composição, invariância, negação.

**Teto medido antes do treino.** A mistura `p_gen·p_vocab + (1−p_gen)·p_copy` só dá massa de cópia a IDs que existem no prompt. Um alvo fora do prompt (`<fim>`, ou o primeiro token do nome sem `Ġ`) só vence o argmax se já for o top-1 da base congelada. A célula 8 mede esse teto por tarefa sob teacher forcing.

Mesmo que todos os gates passem: `ativo_no_chat=false`, `promovido=false`, e a porta conversacional continua pendente.

> **Ambiente de execução → Executar tudo** em um Colab com GPU.


In [ ]:
# Célula 0 — ambiente, GPU, Drive, dependências
import os, sys, gc, json, math, time, hashlib, shutil, traceback, re, itertools, copy, random
from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict

os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("PRECHECK 0/9 — ambiente", flush=True)

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
except Exception as exc:
    raise SystemExit(f"ERRO: PyTorch indisponível: {exc}")

try:
    from google.colab import drive
except Exception as exc:
    raise SystemExit(
        "ERRO: este notebook foi preparado para Google Colab. "
        f"google.colab não está disponível: {exc}"
    )

if not torch.cuda.is_available():
    raise SystemExit(
        "ERRO: GPU não disponível. No Colab: Ambiente de execução → Alterar tipo de ambiente → GPU."
    )

print(f"GPU OK: {torch.cuda.get_device_name(0)}", flush=True)
print(f"PyTorch={torch.__version__} CUDA={torch.version.cuda}", flush=True)
torch.backends.cuda.matmul.allow_tf32 = True

print("Montando Google Drive...", flush=True)
drive.mount("/content/drive", force_remount=False)
print("Drive montado.", flush=True)

try:
    import tokenizers
    from tokenizers import Tokenizer
except Exception:
    import subprocess
    print("Instalando tokenizers...", flush=True)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tokenizers>=0.20,<1"])
    import tokenizers
    from tokenizers import Tokenizer

print(f"tokenizers={tokenizers.__version__}", flush=True)

SEED = 20261012
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
print("PRECHECK 0/9 OK", flush=True)


In [ ]:
# Célula 1 — clone da branch do experimento e verificação dos dados congelados
import subprocess, importlib

print("PRECHECK 1/9 — repositório e dados congelados", flush=True)
REPO_URL = "https://github.com/HROSONE/CRIVO"
REPO_BRANCH = "claude/affectionate-carson-22yoe0"
REPO_DIR = Path("/content/CRIVO")
EXP_REL = Path("experimentos/pointer_copy_causal_v3_20261012")

# SHA-256 dos arquivos congelados antes de qualquer treino (ver manifesto.json no repo).
ESPERADO_SHA = {
    "blind_novo_v3.json": "7a188ddfd948e38193dd55b1749513f63272b1ce04863cda7e6ed42022d984c8",
    "blind_regressao_v2.json": "c8aeec37b4beaa935cdb2c498717308b9dc155a6dbda3a1a0f6cb9801e937756",
    "treino_dev_v3.json": "4fb0edeb47742f7a3eb849e2bab76e22da59cf38c20bf752f3a039af8202edda",
    "dados_v3.py": "43aa9bb78fc183f90213d9ce023a241b46483c6ca44dbbceb8fd393ea0b1118e",
}

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
r = subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
    capture_output=True, text=True,
)
if r.returncode != 0:
    raise SystemExit(f"ERRO ao clonar {REPO_URL}@{REPO_BRANCH}:\n{r.stdout}\n{r.stderr}")

commit = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
EXP_DIR = REPO_DIR / EXP_REL

def sha256_arquivo(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for bloco in iter(lambda: f.read(1024 * 1024), b""):
            h.update(bloco)
    return h.hexdigest()

for nome, esperado in ESPERADO_SHA.items():
    obtido = sha256_arquivo(EXP_DIR / nome)
    if obtido != esperado:
        raise SystemExit(f"ERRO: {nome} difere do congelado.\nesperado={esperado}\nobtido={obtido}")

for p in (str(REPO_DIR), str(EXP_DIR)):
    if p not in sys.path:
        sys.path.insert(0, p)
import linguagem_profunda as lp
import dados_v3 as d3
importlib.reload(d3)

esperados = ["<pad>", "<documento>", "<usuario>", "<assistente>", "<fim>"]
if list(lp.ESPECIAIS) != esperados:
    raise SystemExit(f"ERRO: especiais do repo mudaram: {lp.ESPECIAIS}")

print("commit:", commit, flush=True)
print("dados congelados conferidos por SHA-256:", list(ESPERADO_SHA), flush=True)
print("PRECHECK 1/9 OK", flush=True)


In [ ]:
# Célula 2 — mesma base do v2, conferida por SHA, copiada para o SSD
print("PRECHECK 2/9 — checkpoint", flush=True)

BASE_SHA_ESPERADO = "c0a56b1546495efafde5df99feebfa084fef414d512076d888b40c9828e4eb4e"
DRIVE_CHECKPOINTS = Path("/content/drive/MyDrive/CRIVO/checkpoints")
BASE_RUN = DRIVE_CHECKPOINTS / "escola_curricular_longa" / "run_transferido_20261012"
ORIGEM = BASE_RUN / "05_05_dialogo_estado"

for nome in ("pesos.pt", "tokenizer.json"):
    if not (ORIGEM / nome).is_file():
        raise SystemExit(f"ERRO: não encontrei {ORIGEM / nome}")

LOCAL = Path("/content/crivo_pointer_v3_base")
if LOCAL.exists():
    shutil.rmtree(LOCAL)
LOCAL.mkdir(parents=True)
for nome in ("pesos.pt", "tokenizer.json"):
    print(f"Copiando {nome} para SSD...", flush=True)
    shutil.copy2(ORIGEM / nome, LOCAL / nome)

SHA_PESOS = sha256_arquivo(LOCAL / "pesos.pt")
SHA_TOKENIZER = sha256_arquivo(LOCAL / "tokenizer.json")
if SHA_PESOS != BASE_SHA_ESPERADO:
    raise SystemExit(f"ERRO: a base não é a mesma do v2.\nesperado={BASE_SHA_ESPERADO}\nobtido={SHA_PESOS}")
print("Base:", BASE_RUN, flush=True)
print("pesos.pt sha256:", SHA_PESOS, "(igual ao v2)", flush=True)
print("PRECHECK 2/9 OK", flush=True)


In [ ]:
# Célula 3 — carregar, congelar e fingerprint da base
print("PRECHECK 3/9 — modelo/tokenizer", flush=True)

try:
    modelo, tokenizer, estado = lp.carregar(LOCAL, dispositivo="cpu")
except Exception as exc:
    raise SystemExit(f"ERRO ao carregar checkpoint: {exc}\n{traceback.format_exc()}")

cfg = modelo.config
if cfg.vocabulario != 8192 or cfg.dimensao != 384 or cfg.camadas != 8 or cfg.cabecas != 8:
    print(
        "AVISO: configuração difere do snapshot esperado "
        f"(vocab={cfg.vocabulario}, dim={cfg.dimensao}, camadas={cfg.camadas}, cabecas={cfg.cabecas}).",
        flush=True,
    )

def fingerprint_modulo(mod: nn.Module) -> str:
    h = hashlib.sha256()
    with torch.no_grad():
        for nome, param in sorted(mod.named_parameters(), key=lambda x: x[0]):
            t = param.detach().cpu().contiguous()
            h.update(nome.encode("utf-8"))
            h.update(str(t.dtype).encode("ascii"))
            h.update(str(tuple(t.shape)).encode("ascii"))
            h.update(t.numpy().tobytes())
    return h.hexdigest()

modelo.eval()
for p in modelo.parameters():
    p.requires_grad_(False)
    p.grad = None

N_BASE = sum(p.numel() for p in modelo.parameters())
HASH_ANTES = fingerprint_modulo(modelo)
FINGERPRINT_V2 = "257a4eb41406b9aeeee8f9cf0471c1d8423f769685643200d03c6886a1ea1566"
if HASH_ANTES != FINGERPRINT_V2:
    raise SystemExit(f"ERRO: fingerprint da base difere do v2: {HASH_ANTES}")

ids_especiais = {}
for s in lp.ESPECIAIS:
    tid = tokenizer.token_to_id(s)
    if tid is None:  # noqa
        raise SystemExit(f"ERRO: especial ausente no tokenizer: {s}")
    ids_especiais[s] = int(tid)

ID_PAD = ids_especiais["<pad>"]
ID_USUARIO = ids_especiais["<usuario>"]
ID_ASSISTENTE = ids_especiais["<assistente>"]
ID_FIM = ids_especiais["<fim>"]
ESPECIAIS_SET = set(ids_especiais.values())
PROIBIDOS_GERACAO = [i for i in ESPECIAIS_SET if i != ID_FIM]

dispositivo = torch.device("cuda")
modelo = modelo.to(dispositivo).eval()

print(
    f"Config: vocab={cfg.vocabulario} dim={cfg.dimensao} camadas={cfg.camadas} "
    f"cabecas={cfg.cabecas} contexto={cfg.contexto} dropout={cfg.dropout}",
    flush=True
)
print(f"Base: {N_BASE:,} parâmetros — todos congelados", flush=True)
print("IDs especiais:", ids_especiais, flush=True)
print("base_hash_before:", HASH_ANTES, flush=True)
print("pesos.pt sha256:", SHA_PESOS, flush=True)
print("tokenizer sha256:", SHA_TOKENIZER, flush=True)
print("PRECHECK 3/9 OK", flush=True)


In [ ]:
# Célula 4 — MESMA cabeça pointer/copy do v2 (código idêntico); fonte = SOMENTE prompt original
print("PRECHECK 4/9 — pointer", flush=True)

@torch.no_grad()
def estados_congelados(ids: torch.Tensor) -> torch.Tensor:
    if ids.ndim != 2:
        raise ValueError("ids deve ser [B,T]")
    if ids.shape[1] > cfg.contexto:
        raise ValueError(f"sequência {ids.shape[1]} > contexto {cfg.contexto}")
    pos = torch.arange(ids.shape[1], device=ids.device)
    x = modelo.embedding(ids) + modelo.posicao(pos)
    for bloco in modelo.blocos:
        x = bloco(x)
    return modelo.norm(x)

def mascara_fonte(ids: torch.Tensor, prompt_lens: torch.Tensor) -> torch.Tensor:
    """
    True somente nas posições do prompt ORIGINAL e que não são especiais.
    Tokens da resposta (teacher-forced ou gerados) nunca entram na fonte de cópia.
    """
    b, t = ids.shape
    pos = torch.arange(t, device=ids.device).unsqueeze(0)
    src = pos < prompt_lens.unsqueeze(1)
    for sid in ESPECIAIS_SET:
        src &= ids.ne(sid)
    return src

class PointerCopy(nn.Module):
    def __init__(self, dimensao: int, vocabulario: int, rank: int = 32, p_gen_init: float = 0.90):
        super().__init__()
        if not (0.0 < p_gen_init < 1.0):
            raise ValueError("p_gen_init deve estar entre 0 e 1")
        self.rank = int(rank)
        self.vocabulario = int(vocabulario)
        self.q = nn.Linear(dimensao, rank, bias=False)
        self.k = nn.Linear(dimensao, rank, bias=False)
        self.gate = nn.Linear(dimensao, 1)
        nn.init.normal_(self.q.weight, std=0.02)
        nn.init.normal_(self.k.weight, std=0.02)
        nn.init.zeros_(self.gate.weight)
        nn.init.constant_(self.gate.bias, math.log(p_gen_init / (1.0 - p_gen_init)))

    def forward(self, h: torch.Tensor, ids: torch.Tensor, source_mask: torch.Tensor):
        b, t, _ = h.shape
        if ids.shape != (b, t) or source_mask.shape != (b, t):
            raise ValueError("shape incompatível em PointerCopy.forward")

        q = self.q(h)
        k = self.k(h)
        scores = torch.matmul(q, k.transpose(-1, -2)) / math.sqrt(self.rank)

        causal = torch.tril(torch.ones((t, t), dtype=torch.bool, device=h.device))
        valido = causal.unsqueeze(0) & source_mask.unsqueeze(1)  # [B,T,T]

        # -1e4 evita NaN em linhas sem nenhuma fonte; depois essas linhas são zeradas.
        scores = scores.masked_fill(~valido, -1e4)
        attn = torch.softmax(scores, dim=-1)
        tem_fonte = valido.any(dim=-1, keepdim=True)
        attn = torch.where(tem_fonte, attn, torch.zeros_like(attn))
        attn = attn * valido.to(attn.dtype)
        attn = attn / attn.sum(dim=-1, keepdim=True).clamp_min(1e-8)

        p_copy = torch.zeros(
            (b, t, self.vocabulario), dtype=h.dtype, device=h.device
        )
        ids_expand = ids.unsqueeze(1).expand(b, t, t)
        p_copy.scatter_add_(2, ids_expand, attn)

        # Defesa redundante: nenhum especial pode receber massa de copy.
        p_copy[:, :, list(ESPECIAIS_SET)] = 0.0
        soma = p_copy.sum(dim=-1, keepdim=True)
        p_copy = torch.where(
            soma > 0,
            p_copy / soma.clamp_min(1e-8),
            torch.zeros_like(p_copy),
        )

        p_gen = torch.sigmoid(self.gate(h))
        return p_copy, p_gen, attn, tem_fonte

def distribuicao_final(pointer_obj: PointerCopy, ids: torch.Tensor, prompt_lens: torch.Tensor):
    modelo.eval()
    with torch.no_grad():
        h = estados_congelados(ids)
        logits = F.linear(h, modelo.embedding.weight)
        p_vocab = torch.softmax(logits, dim=-1)
        if PROIBIDOS_GERACAO:
            p_vocab = p_vocab.clone()
            p_vocab[:, :, PROIBIDOS_GERACAO] = 0.0
            p_vocab = p_vocab / p_vocab.sum(dim=-1, keepdim=True).clamp_min(1e-8)

    h = h.detach()
    src = mascara_fonte(ids, prompt_lens)
    p_copy, p_gen, attn, tem_fonte = pointer_obj(h, ids, src)

    # Sem nenhuma fonte copiável, força a rota vocab (não renormaliza um vetor zero).
    p_gen_eff = torch.where(tem_fonte, p_gen, torch.ones_like(p_gen))
    p_final = p_gen_eff * p_vocab + (1.0 - p_gen_eff) * p_copy
    p_final = p_final / p_final.sum(dim=-1, keepdim=True).clamp_min(1e-8)
    return p_final, p_vocab, p_copy, p_gen, p_gen_eff, attn, src

RANK = 32
pointer = PointerCopy(cfg.dimensao, cfg.vocabulario, rank=RANK).to(dispositivo)
N_POINTER = sum(p.numel() for p in pointer.parameters() if p.requires_grad)

# Teste estrutural independente de dados reais.
_b, _t = 2, 7
_h = torch.randn(_b, _t, cfg.dimensao, device=dispositivo)
_ids = torch.randint(5, cfg.vocabulario, (_b, _t), device=dispositivo)
_ids[:, 0] = ID_USUARIO
_pm = torch.tensor([5, 4], device=dispositivo)
_src = mascara_fonte(_ids, _pm)
_pc, _pg, _a, _tf = pointer(_h, _ids, _src)
assert _pc.shape == (_b, _t, cfg.vocabulario)
assert _a.shape == (_b, _t, _t)
assert torch.isfinite(_pc).all() and torch.isfinite(_a).all()
assert float(_pc[:, :, list(ESPECIAIS_SET)].abs().max().item()) == 0.0
del _h, _ids, _pm, _src, _pc, _pg, _a, _tf

print(f"Pointer/gate treinável: {N_POINTER:,} parâmetros ({100*N_POINTER/N_BASE:.4f}% da base)", flush=True)
print(f"p_gen inicial={torch.sigmoid(pointer.gate.bias.detach()).item():.4f}", flush=True)
print("PRECHECK 4/9 OK", flush=True)


In [ ]:
# Célula 5 — dados congelados; re-tokenização no runtime tem de bater byte a byte
print("PRECHECK 5/9 — dados", flush=True)

def ler_json(nome):
    return json.loads((EXP_DIR / nome).read_text(encoding="utf-8"))

BLIND_NOVO = ler_json("blind_novo_v3.json")
BLIND_V2 = ler_json("blind_regressao_v2.json")
TREINO_DEV = ler_json("treino_dev_v3.json")

def remontar(entidades):
    casos = []
    for i, ent in enumerate(entidades):
        for j, tarefa in enumerate(d3.TAREFAS):
            casos.append(d3.montar_caso(lp, tokenizer, ent, tarefa, d3.POSICOES[(i + j) % 3], cfg.contexto))
    return casos

def conferir_igual(montados, congelados, rotulo):
    if len(montados) != len(congelados):
        raise SystemExit(f"ERRO: {rotulo} tem {len(montados)} casos; congelado tem {len(congelados)}")
    for a, b in zip(montados, congelados):
        if a["prompt_ids"] != b["prompt_ids"] or a["resp_ids"] != b["resp_ids"]:
            raise SystemExit(f"ERRO: tokenização do runtime divergiu do congelado em {rotulo}: {a['nome']}/{a['tarefa']}")

BLIND_NOVO_EX = remontar(BLIND_NOVO["entidades"])
conferir_igual(BLIND_NOVO_EX, BLIND_NOVO["casos"], "blind novo")
BLIND_V2_REAL_EX = remontar(BLIND_V2["entidades"])
conferir_igual(BLIND_V2_REAL_EX, BLIND_V2["casos_formato_real"], "blind v2 formato real")
BLIND_V2_LEGADO_EX = BLIND_V2["casos_legado_v2"]  # formato antigo (IDs colados) só para regressão

rr = random.Random(SEED + 2002)
TREINO = []
for i, ent in enumerate(TREINO_DEV["treino"]):
    for j, tarefa in enumerate(d3.TAREFAS):
        pos = d3.POSICOES[(i + j + rr.randrange(3)) % 3]
        TREINO.append(d3.montar_caso(lp, tokenizer, ent, tarefa, pos, cfg.contexto))
rr.shuffle(TREINO)

DEV_EX = []
for i, ent in enumerate(TREINO_DEV["dev"]):
    for j, tarefa in enumerate(("copia", "categoria", "funcao")):
        DEV_EX.append(d3.montar_caso(lp, tokenizer, ent, tarefa, d3.POSICOES[(i + j) % 3], cfg.contexto))

NO_COPY_EX = [d3.montar_no_copy(lp, tokenizer, q, a, cfg.contexto) for q, a in d3.NO_COPY_PARES]

def nomes(xs):
    return {x["nome"].casefold() for x in xs}

conjuntos = {
    "treino": TREINO_DEV["treino"], "dev": TREINO_DEV["dev"],
    "blind_v2": BLIND_V2["entidades"], "blind_novo": BLIND_NOVO["entidades"],
}
for a in conjuntos:
    for b in conjuntos:
        if a < b and nomes(conjuntos[a]) & nomes(conjuntos[b]):
            raise SystemExit(f"ERRO: interseção entre {a} e {b}")

def resumo_split(xs):
    return {"n": len(xs), "grupo": dict(Counter(x["grupo"] for x in xs)),
            "tokens": dict(sorted(Counter(x["n_tokens"] for x in xs).items()))}

def copiabilidade(casos):
    out = {}
    for t in d3.TAREFAS:
        xs = [c for c in casos if c["tarefa"] == t]
        out[t] = {
            "primeiro_token_copiavel": sum(c["copy_resp"][c["nome_resp"].index(True)] for c in xs) / len(xs),
            "nome_inteiro_copiavel": sum(c["nome_todo_copiavel"] for c in xs) / len(xs),
        }
    return out

COPIABILIDADE = copiabilidade(BLIND_NOVO_EX)
print("treino:", len(TREINO), "dev:", len(DEV_EX), "no-copy:", len(NO_COPY_EX), flush=True)
print("blind novo:", len(BLIND_NOVO_EX), "| blind v2 real:", len(BLIND_V2_REAL_EX),
      "| blind v2 legado:", len(BLIND_V2_LEGADO_EX), flush=True)
for k, v in conjuntos.items():
    print(f"  {k}:", resumo_split(v), flush=True)
print("Copiabilidade por ID no blind novo:", json.dumps(COPIABILIDADE, ensure_ascii=False), flush=True)
print("Interseções entre treino/dev/blind_v2/blind_novo = 0", flush=True)
print("PRECHECK 5/9 OK", flush=True)


In [ ]:
# Célula 6 — batch, perda, geração (código do v2) e avaliação v3
print("PRECHECK 6/9 — perda/geração/avaliação", flush=True)

def colar_batch(exemplos, device):
    fulls, labels_rows, copy_rows, prompt_lens, fim_rows = [], [], [], [], []
    for ex in exemplos:
        prompt = list(ex["prompt_ids"])
        resp = list(ex["resp_ids"])
        copy_resp = list(ex["copy_resp"])
        if len(resp) != len(copy_resp):
            raise ValueError("resp_ids/copy_resp com tamanhos diferentes")
        full = prompt + resp
        if len(full) > cfg.contexto:
            raise ValueError("batch contém sequência acima do contexto")

        labels = [-100] * len(prompt) + resp
        copy_target = [False] * len(prompt) + copy_resp
        fim_target = [False] * len(prompt) + [
            ex["tarefa"] == "copia" and i == len(resp) - 1 and tok == ID_FIM
            for i, tok in enumerate(resp)
        ]

        fulls.append(full)
        labels_rows.append(labels)
        copy_rows.append(copy_target)
        fim_rows.append(fim_target)
        prompt_lens.append(len(prompt))

    tmax = max(map(len, fulls))
    b = len(fulls)
    ids = torch.full((b, tmax), ID_PAD, dtype=torch.long, device=device)
    labels = torch.full((b, tmax), -100, dtype=torch.long, device=device)
    copy_targets = torch.zeros((b, tmax), dtype=torch.bool, device=device)
    fim_targets = torch.zeros((b, tmax), dtype=torch.bool, device=device)

    for i, (full, lab, cpy, fim) in enumerate(zip(fulls, labels_rows, copy_rows, fim_rows)):
        n = len(full)
        ids[i, :n] = torch.tensor(full, dtype=torch.long, device=device)
        labels[i, :n] = torch.tensor(lab, dtype=torch.long, device=device)
        copy_targets[i, :n] = torch.tensor(cpy, dtype=torch.bool, device=device)
        fim_targets[i, :n] = torch.tensor(fim, dtype=torch.bool, device=device)

    prompt_lens_t = torch.tensor(prompt_lens, dtype=torch.long, device=device)
    return ids, labels, copy_targets, prompt_lens_t, fim_targets

def perda_batch(pointer_obj, exemplos, device):
    ids, labels, copy_targets, prompt_lens, fim_targets = colar_batch(exemplos, device)
    p_final, _, p_copy, p_gen, _, _, src = distribuicao_final(pointer_obj, ids, prompt_lens)

    # hidden position t predicts token t+1
    pred = p_final[:, :-1, :]
    copy_pred = p_copy[:, :-1, :]
    gate_pred = p_gen[:, :-1, 0]
    gold = labels[:, 1:]
    supervised = gold.ne(-100)
    copy_gold_mask = copy_targets[:, 1:] & supervised

    if not supervised.any():
        raise RuntimeError("batch sem tokens supervisionados")

    safe_gold = gold.clamp_min(0)
    prob_gold = pred.gather(-1, safe_gold.unsqueeze(-1)).squeeze(-1)
    mix_nll = -torch.log(prob_gold.clamp_min(1e-8))
    loss_mix = mix_nll[supervised].mean()

    # Copy CE SOMENTE nos tokens explicitamente marcados como span da entidade.
    if copy_gold_mask.any():
        copy_prob_gold = copy_pred.gather(-1, safe_gold.unsqueeze(-1)).squeeze(-1)
        copy_nll = -torch.log(copy_prob_gold.clamp_min(1e-8))
        loss_copy = copy_nll[copy_gold_mask].mean()
        # Se isso for zero, há bug de source mask/span.
        if not torch.isfinite(loss_copy):
            raise RuntimeError("loss_copy não finita")
    else:
        loss_copy = torch.zeros((), device=device)

    # Gate supervisionado: copy span -> p_gen=0; demais tokens da resposta -> p_gen=1.
    gate_target = (~copy_gold_mask).float()
    gate_bce_all = F.binary_cross_entropy(
        gate_pred.clamp(1e-6, 1 - 1e-6), gate_target, reduction="none"
    )
    loss_gate = gate_bce_all[supervised].mean()

    # v3: <fim> da cópia pura supervisionado explicitamente (copiar = nome inteiro e parar).
    fim_gold_mask = fim_targets[:, 1:] & supervised
    if fim_gold_mask.any():
        loss_fim = mix_nll[fim_gold_mask].mean()
    else:
        loss_fim = torch.zeros((), device=device)

    loss = loss_mix + 0.50 * loss_copy + 0.10 * loss_gate + 0.50 * loss_fim

    with torch.no_grad():
        pg_copy = gate_pred[copy_gold_mask].mean().item() if copy_gold_mask.any() else None
        gen_mask = supervised & ~copy_gold_mask
        pg_gen = gate_pred[gen_mask].mean().item() if gen_mask.any() else None
        src_frac = src.float().mean().item()

    metricas = {
        "loss_mix": float(loss_mix.detach().item()),
        "loss_copy": float(loss_copy.detach().item()),
        "loss_gate": float(loss_gate.detach().item()),
        "loss_fim": float(loss_fim.detach().item()),
        "p_gen_copy": None if pg_copy is None else float(pg_copy),
        "p_gen_generate": None if pg_gen is None else float(pg_gen),
        "source_mask_fraction": float(src_frac),
    }
    return loss, metricas

@torch.no_grad()
def probs_base(ids):
    h = estados_congelados(ids)
    logits = F.linear(h, modelo.embedding.weight)
    p = torch.softmax(logits, dim=-1)
    if PROIBIDOS_GERACAO:
        p[:, :, PROIBIDOS_GERACAO] = 0.0
        p = p / p.sum(dim=-1, keepdim=True).clamp_min(1e-8)
    return p

@torch.no_grad()
def gerar_base(prompt_ids, max_novos=28):
    ids = list(prompt_ids)
    gerados = []
    limite = min(max_novos, max(0, cfg.contexto - len(ids)))
    for _ in range(limite):
        x = torch.tensor([ids], dtype=torch.long, device=dispositivo)
        p = probs_base(x)[0, -1]
        nxt = int(torch.argmax(p).item())
        if nxt == ID_FIM:
            return gerados, True
        if nxt in PROIBIDOS_GERACAO:
            raise RuntimeError("base tentou emitir token especial proibido após mascaramento")
        gerados.append(nxt)
        ids.append(nxt)
    return gerados, False

@torch.no_grad()
def gerar_pointer(pointer_obj, prompt_ids, max_novos=28):
    ids = list(prompt_ids)
    prompt_len = len(prompt_ids)
    gerados = []
    pgens = []
    limite = min(max_novos, max(0, cfg.contexto - prompt_len))
    for _ in range(limite):
        x = torch.tensor([ids], dtype=torch.long, device=dispositivo)
        pl = torch.tensor([prompt_len], dtype=torch.long, device=dispositivo)
        p_final, _, _, p_gen, _, _, src = distribuicao_final(pointer_obj, x, pl)

        # Teste anti-vazamento: qualquer token gerado fica FORA da source mask.
        if len(ids) > prompt_len and bool(src[0, prompt_len:].any().item()):
            raise RuntimeError("BUG: token gerado entrou na source mask do pointer")

        pgens.append(float(p_gen[0, -1, 0].item()))
        nxt = int(torch.argmax(p_final[0, -1]).item())
        if nxt == ID_FIM:
            return gerados, True, (sum(pgens) / len(pgens) if pgens else None)
        if nxt in PROIBIDOS_GERACAO:
            raise RuntimeError("pointer tentou emitir especial proibido")
        gerados.append(nxt)
        ids.append(nxt)
    return gerados, False, (sum(pgens) / len(pgens) if pgens else None)


def max_novos_ex(ex):
    return {
        "copia": 8, "categoria": 18, "funcao": 22, "invariancia": 24,
        "composicao": 28, "negacao": 28, "no_copy": 12,
    }.get(ex["tarefa"], 24)

def avaliar_conjunto(modo, casos, pointer_obj=None, rotulo="eval"):
    """Formato real: métricas no texto decodificado (dados_v3.avaliar_saida)."""
    if modo not in ("base", "pointer"):
        raise ValueError(modo)
    print(f"Avaliando {rotulo}: {len(casos)} casos", flush=True)
    linhas, t0 = [], time.time()
    for i, ex in enumerate(casos, 1):
        if modo == "base":
            gerados, terminou = gerar_base(ex["prompt_ids"], max_novos=max_novos_ex(ex))
            pg = None
        else:
            gerados, terminou, pg = gerar_pointer(pointer_obj, ex["prompt_ids"], max_novos=max_novos_ex(ex))
        texto = tokenizer.decode(gerados)
        linha = {k: ex[k] for k in ("nome", "grupo", "n_tokens", "tarefa", "posicao",
                                    "nome_no_inicio_da_resposta", "nome_todo_copiavel")}
        linha.update(d3.avaliar_saida(ex, texto, terminou))
        linha.update({"p_gen_medio": pg, "alvo": ex["resposta"], "saida": texto.strip()})
        linhas.append(linha)
        if i == 1 or i % 100 == 0 or i == len(casos):
            print(f"  {rotulo}: {i}/{len(casos)} ({time.time()-t0:.1f}s)", flush=True)
    return linhas

def contem_subspan(hay, needle):
    n = len(needle)
    return bool(needle) and any(hay[i:i+n] == needle for i in range(len(hay) - n + 1))

def avaliar_legado(modo, casos, pointer_obj=None, rotulo="legado"):
    """Formato antigo do v2 (IDs colados) com as métricas do v2, mais nome+<fim>."""
    print(f"Avaliando {rotulo}: {len(casos)} casos", flush=True)
    linhas, t0 = [], time.time()
    for i, ex in enumerate(casos, 1):
        if modo == "base":
            gerados, terminou = gerar_base(ex["prompt_ids"], max_novos=max_novos_ex(ex))
        else:
            gerados, terminou, _ = gerar_pointer(pointer_obj, ex["prompt_ids"], max_novos=max_novos_ex(ex))
        nome_ok = contem_subspan(gerados, ex["nome_ids"])
        cat_ok = contem_subspan(gerados, ex["cat_ids"])
        fun_ok = contem_subspan(gerados, ex["fun_ids"])
        txt = tokenizer.decode(gerados).casefold()
        neg = ("não" in txt) or ("nao" in txt)
        sem = {"copia": gerados == ex["nome_ids"], "categoria": nome_ok and cat_ok,
               "funcao": nome_ok and fun_ok, "invariancia": nome_ok and neg,
               "composicao": nome_ok and cat_ok and fun_ok, "negacao": nome_ok and cat_ok and neg}[ex["tarefa"]]
        linhas.append({"tarefa": ex["tarefa"], "grupo": ex["grupo"], "entity_exact": nome_ok,
                       "pure_copy_exact": ex["tarefa"] == "copia" and gerados == ex["nome_ids"],
                       "pure_copy_exact_fim": ex["tarefa"] == "copia" and gerados == ex["nome_ids"] and terminou,
                       "semantic_ok": bool(sem), "terminou": bool(terminou),
                       "saida": tokenizer.decode(gerados)})
        if i == 1 or i % 100 == 0 or i == len(casos):
            print(f"  {rotulo}: {i}/{len(casos)} ({time.time()-t0:.1f}s)", flush=True)
    return linhas

def resumir_legado(linhas):
    def taxa(xs, k):
        return None if not xs else sum(bool(x[k]) for x in xs) / len(xs)
    copia = [r for r in linhas if r["tarefa"] == "copia"]
    out = {"n": len(linhas), "exact_entity_match": taxa(linhas, "entity_exact"),
           "pure_copy_exact_match": taxa(copia, "pure_copy_exact"),
           "pure_copy_exact_fim": taxa(copia, "pure_copy_exact_fim"),
           "lexical_close": taxa([r for r in linhas if r["grupo"] == "close"], "entity_exact"),
           "lexical_distant": taxa([r for r in linhas if r["grupo"] == "distant"], "entity_exact"),
           "termination_rate": taxa(linhas, "terminou")}
    for t in ("categoria", "funcao", "invariancia", "composicao", "negacao"):
        out[t] = taxa([r for r in linhas if r["tarefa"] == t], "semantic_ok")
    out["semantica_geral"] = sum(out[t] for t in ("categoria", "funcao", "invariancia", "composicao", "negacao")) / 5
    return out

@torch.no_grad()
def teto_teacher_forcing(casos):
    """Limite superior exato da mistura p_gen*p_vocab + (1-p_gen)*p_copy sob teacher forcing.

    Um token que não está no prompt (ex.: <fim>, ou o primeiro token do nome sem 'Ġ')
    recebe só g*p_vocab; qualquer token com p_vocab maior recebe pelo menos g*p_vocab
    também, logo esse alvo só vence o argmax se já for o top-1 da base congelada.
    """
    linhas = []
    for ex in casos:
        prompt, resp = ex["prompt_ids"], ex["resp_ids"]
        x = torch.tensor([prompt + resp], dtype=torch.long, device=dispositivo)
        p = probs_base(x)[0]
        P = len(prompt)
        top1 = p[P - 1:P - 1 + len(resp)].argmax(-1).tolist()
        fonte = {t for t in prompt if t not in ESPECIAIS_SET}
        alc = [(t in fonte) or (top1[i] == t) for i, t in enumerate(resp)]
        nome_pos = [i for i, m in enumerate(ex["nome_resp"]) if m]
        linhas.append({
            "tarefa": ex["tarefa"],
            "nome_alcancavel": all(alc[i] for i in nome_pos),
            "resposta_alcancavel": all(alc),
            "fim_top1_base": top1[-1] == ID_FIM,
            "primeiro_nome_top1_base": top1[nome_pos[0]] == resp[nome_pos[0]],
        })
    out = {}
    for t in d3.TAREFAS:
        xs = [r for r in linhas if r["tarefa"] == t]
        out[t] = {k: sum(r[k] for r in xs) / len(xs)
                  for k in ("nome_alcancavel", "resposta_alcancavel", "fim_top1_base", "primeiro_nome_top1_base")}
    out["copia"]["teto_exact_name_fim"] = out["copia"]["resposta_alcancavel"]
    return out

print("PRECHECK 6/9 OK", flush=True)


In [ ]:
# Célula 7 — SANITY + smoke test adversarial antes de qualquer treino real
print("PRECHECK 7/9 — SANITY/SMOKE", flush=True)

# Estrutura dos dados
assert len(TREINO_DEV["treino"]) == 480 and len(TREINO_DEV["dev"]) == 80
assert len(BLIND_NOVO_EX) == 600 and len(BLIND_V2_REAL_EX) == 600 and len(BLIND_V2_LEGADO_EX) == 600
assert all(ex["prompt_ids"][-2:] == [ID_FIM, ID_ASSISTENTE] for ex in BLIND_NOVO_EX + TREINO[:200])
assert all(ex["resp_ids"][-1] == ID_FIM for ex in BLIND_NOVO_EX + TREINO)
assert all(len(ex["prompt_ids"]) + len(ex["resp_ids"]) <= cfg.contexto for ex in BLIND_NOVO_EX + TREINO)
# <fim> da cópia pura entra na perda explícita
_ids, _lab, _cp, _pl, _fim = colar_batch([x for x in TREINO if x["tarefa"] == "copia"][:4], dispositivo)
assert int(_fim.sum().item()) == 4, "máscara do <fim> da cópia pura não marcou 1 posição por exemplo"
del _ids, _lab, _cp, _pl, _fim

pointer_sanity = PointerCopy(cfg.dimensao, cfg.vocabulario, rank=RANK).to(dispositivo)
opt_sanity = torch.optim.AdamW(pointer_sanity.parameters(), lr=1e-3, weight_decay=0.0)

rr_s = random.Random(SEED + 3003)
copy_pool = [x for x in TREINO if x["tarefa"] in ("copia", "categoria", "funcao")]
no_copy_pool = NO_COPY_EX

for passo in range(1, 6):
    batch = rr_s.sample(copy_pool, 8) + [rr_s.choice(no_copy_pool) for _ in range(4)]
    rr_s.shuffle(batch)
    opt_sanity.zero_grad(set_to_none=True)
    loss_s, met_s = perda_batch(pointer_sanity, batch, dispositivo)
    if not torch.isfinite(loss_s):
        raise SystemExit(f"ERRO SANITY: loss não finita no passo {passo}")
    loss_s.backward()

    soma_grad = 0.0
    for nome, p in pointer_sanity.named_parameters():
        if p.grad is None:
            raise SystemExit(f"ERRO SANITY: pointer sem gradiente em {nome}")
        if not torch.isfinite(p.grad).all():
            raise SystemExit(f"ERRO SANITY: gradiente NaN/Inf em {nome}")
        soma_grad += float(p.grad.abs().sum().item())

    if soma_grad <= 0:
        raise SystemExit("ERRO SANITY: todos os gradientes do pointer são zero")

    for nome, p in modelo.named_parameters():
        if p.requires_grad:
            raise SystemExit(f"ERRO SANITY: base descongelada em {nome}")
        if p.grad is not None:
            raise SystemExit(f"ERRO SANITY: base recebeu gradiente em {nome}")

    torch.nn.utils.clip_grad_norm_(pointer_sanity.parameters(), 1.0)
    opt_sanity.step()
    print(
        f"  sanity step {passo}/5 loss={loss_s.item():.4f} "
        f"mix={met_s['loss_mix']:.4f} copy={met_s['loss_copy']:.4f} "
        f"gate={met_s['loss_gate']:.4f} fim={met_s['loss_fim']:.4f}",
        flush=True,
    )

# Geração autoregressiva real para pegar shape/source-mask bugs que teacher forcing não pega.
teste_ex = next(x for x in DEV_EX if x["tarefa"] == "copia")
gerados_s, _, pg_s = gerar_pointer(pointer_sanity, teste_ex["prompt_ids"], max_novos=8)
print("  smoke geração:", tokenizer.decode(gerados_s), "p_gen_médio=", pg_s, flush=True)

HASH_SANITY = fingerprint_modulo(modelo)
if HASH_SANITY != HASH_ANTES:
    raise SystemExit(
        "ERRO SANITY: hash da base mudou.\n"
        f"antes={HASH_ANTES}\ndepois={HASH_SANITY}"
    )

del pointer_sanity, opt_sanity
torch.cuda.empty_cache()

# Recria pointer LIMPO: os 5 steps acima não contaminam o experimento.
pointer = PointerCopy(cfg.dimensao, cfg.vocabulario, rank=RANK).to(dispositivo)
print("SANITY CHECK PASSOU", flush=True)
print("Base intacta; gradiente só no pointer; source-mask bloqueia resposta; geração não quebrou.", flush=True)
print("Cabeça recriada do zero (nada do pointer_head.pt do v2 é carregado).", flush=True)
print("\n✅ PRECHECK COMPLETO — pronto para medir teto, baseline e treinar.", flush=True)


In [ ]:
# Célula 8 — teto da arquitetura + BASELINE congelado, antes de qualquer treino
print("PRECHECK 8/9 — teto e baseline", flush=True)

TETO = teto_teacher_forcing(BLIND_NOVO_EX)
print("\n=== TETO (teacher forcing, base congelada) no blind novo ===", flush=True)
print(json.dumps(TETO, ensure_ascii=False, indent=2), flush=True)
print(
    f"Com esta arquitetura, exact-name+<fim> na cópia pura não pode passar de "
    f"{100*TETO['copia']['teto_exact_name_fim']:.1f}% sob teacher forcing.",
    flush=True,
)

print("\n=== BASELINE — base congelada sem pointer ===", flush=True)
BASE_NOVO_LINHAS = avaliar_conjunto("base", BLIND_NOVO_EX, rotulo="base/blind_novo")
BASE_NOVO = d3.resumir(BASE_NOVO_LINHAS)
BASE_V2REAL_LINHAS = avaliar_conjunto("base", BLIND_V2_REAL_EX, rotulo="base/blind_v2_real")
BASE_V2REAL = d3.resumir(BASE_V2REAL_LINHAS)
BASE_LEGADO = resumir_legado(avaliar_legado("base", BLIND_V2_LEGADO_EX, rotulo="base/blind_v2_legado"))
print("blind novo:", json.dumps(BASE_NOVO, ensure_ascii=False, indent=2), flush=True)
print("blind v2 legado (deve reproduzir o baseline do v2):", json.dumps(BASE_LEGADO, ensure_ascii=False, indent=2), flush=True)
print("PRECHECK 8/9 OK", flush=True)


In [ ]:
# Célula 9 — treino da cabeça REINICIALIZADA: mesmos hiperparâmetros do v2
PASSOS = 1000
BATCH = 16
N_NO_COPY = 4          # 25% do batch protege p_gen / geração livre
LR = 2e-3
EVAL_CADA = 100
PACIENCIA = 4

rr_train = random.Random(SEED + 4004)
otimizador = torch.optim.AdamW(pointer.parameters(), lr=LR, weight_decay=0.01)

dev_copy = [x for x in DEV_EX if x["tarefa"] == "copia"][:16]
dev_func = [x for x in DEV_EX if x["tarefa"] == "funcao"][:16]

@torch.no_grad()
def preservacao_next_token(pointer_obj):
    """Igual ao v2: primeiro token de respostas genéricas, base vs pointer."""
    prompts = [
        "Explique em uma frase o que é gravidade.", "Qual é a função de uma cadeira?",
        "O que significa memória?", "Diga uma característica da água.", "O que é uma estrela?",
        "Qual é a função de um recipiente?", "Explique o que é temperatura.",
        "O que significa aprender?", "Diga uma função de uma ferramenta.", "O que é uma hipótese?",
        "Explique o que é energia.", "Qual é a diferença entre fato e opinião?",
    ]
    acordos, tvs = [], []
    for msg in prompts:
        prompt = lp.fonte_dialogo(tokenizer, msg, (), cfg.contexto)
        x = torch.tensor([prompt], dtype=torch.long, device=dispositivo)
        pb = probs_base(x)[0, -1]
        pl = torch.tensor([len(prompt)], dtype=torch.long, device=dispositivo)
        pp = distribuicao_final(pointer_obj, x, pl)[0][0, -1]
        acordos.append(int(torch.argmax(pb)) == int(torch.argmax(pp)))
        tvs.append(0.5 * torch.abs(pb - pp).sum().item())
    return {"top1_agreement": sum(acordos) / len(acordos),
            "mean_total_variation": sum(tvs) / len(tvs), "n": len(prompts)}

@torch.no_grad()
def dev_monitor(pointer_obj):
    pointer_obj.eval()
    ac_copy = 0
    for ex in dev_copy:  # v3: acerto = nome inteiro e <fim>
        g, fim, _ = gerar_pointer(pointer_obj, ex["prompt_ids"], max_novos=8)
        ac_copy += d3.avaliar_saida(ex, tokenizer.decode(g), fim)["exact_name_fim"]
    copy_acc = ac_copy / max(1, len(dev_copy))
    ac_func = 0
    for ex in dev_func:
        g, fim, _ = gerar_pointer(pointer_obj, ex["prompt_ids"], max_novos=22)
        ac_func += d3.avaliar_saida(ex, tokenizer.decode(g), fim)["semantic_ok"]
    func_acc = ac_func / max(1, len(dev_func))
    pres = preservacao_next_token(pointer_obj)
    score = copy_acc + 0.25 * func_acc + 0.25 * pres["top1_agreement"]
    pointer_obj.train()
    return {"copy_exact_fim": copy_acc, "funcao": func_acc,
            "preservation_top1": pres["top1_agreement"],
            "preservation_tv": pres["mean_total_variation"], "score": score}

historico = []
melhor_score = -1.0
melhor_state = None
melhor_passo = None
sem_melhora = 0
passos_feitos = 0
t0 = time.time()

entity_batch_n = BATCH - N_NO_COPY
print(
    f"Iniciando treino real: até {PASSOS} passos | batch={BATCH} "
    f"({entity_batch_n} entidade + {N_NO_COPY} no-copy) | lr={LR}",
    flush=True,
)

pointer.train()
for passo in range(1, PASSOS + 1):
    batch = rr_train.sample(TREINO, entity_batch_n)
    batch += [rr_train.choice(NO_COPY_EX) for _ in range(N_NO_COPY)]
    rr_train.shuffle(batch)

    otimizador.zero_grad(set_to_none=True)
    loss, met = perda_batch(pointer, batch, dispositivo)
    if not torch.isfinite(loss):
        raise SystemExit(f"ERRO: loss NaN/Inf no passo {passo}")

    loss.backward()
    for nome, p in pointer.named_parameters():
        if p.grad is None or not torch.isfinite(p.grad).all():
            raise SystemExit(f"ERRO: gradiente inválido no pointer: {nome}")

    torch.nn.utils.clip_grad_norm_(pointer.parameters(), 1.0)
    otimizador.step()
    passos_feitos = passo

    if passo == 1 or passo % 25 == 0:
        print(
            f"PASSO {passo:4d}/{PASSOS} "
            f"loss={loss.item():.4f} mix={met['loss_mix']:.4f} "
            f"copy={met['loss_copy']:.4f} gate={met['loss_gate']:.4f} fim={met['loss_fim']:.4f} "
            f"p_gen(copy)={met['p_gen_copy']} p_gen(gen)={met['p_gen_generate']}",
            flush=True,
        )

    if passo == 1 or passo % EVAL_CADA == 0:
        dm = dev_monitor(pointer)
        registro = {
            "passo": passo,
            "loss": float(loss.item()),
            **met,
            **{f"dev_{k}": float(v) for k, v in dm.items()},
            "segundos": round(time.time() - t0, 1),
        }
        historico.append(registro)
        print(
            f"  DEV passo={passo} copia+fim={dm['copy_exact_fim']:.3f} "
            f"funcao={dm['funcao']:.3f} preserve_top1={dm['preservation_top1']:.3f} "
            f"TV={dm['preservation_tv']:.3f} score={dm['score']:.3f}",
            flush=True,
        )

        if dm["score"] > melhor_score + 1e-4:
            melhor_score = dm["score"]
            melhor_state = {k: v.detach().cpu().clone() for k, v in pointer.state_dict().items()}
            melhor_passo = passo
            sem_melhora = 0
        else:
            sem_melhora += 1
            print(f"  sem melhora composta: {sem_melhora}/{PACIENCIA}", flush=True)

        if passo >= 300 and sem_melhora >= PACIENCIA:
            print("EARLY STOP: score composto do dev parou de melhorar.", flush=True)
            break

    if passo % 200 == 0:
        h = fingerprint_modulo(modelo)
        if h != HASH_ANTES:
            raise SystemExit("ERRO: hash da base mudou durante o treino. Experimento inválido.")

if melhor_state is None:
    raise SystemExit("ERRO: nenhum estado válido do pointer foi registrado.")

pointer.load_state_dict(melhor_state)
pointer.eval()

HASH_APOS_TREINO = fingerprint_modulo(modelo)
if HASH_APOS_TREINO != HASH_ANTES:
    raise SystemExit(
        "ERRO: Transformer mudou durante o treino.\n"
        f"antes={HASH_ANTES}\ndepois={HASH_APOS_TREINO}"
    )

print(
    f"Treino finalizado: {passos_feitos} passos executados; "
    f"melhor_score_dev={melhor_score:.4f} no passo {melhor_passo}; base intacta.",
    flush=True,
)


In [ ]:
# Célula 10 — avaliação FINAL nos mesmos blinds congelados
print("\n=== POINTER v3 FINAL ===", flush=True)
FINAL_NOVO_LINHAS = avaliar_conjunto("pointer", BLIND_NOVO_EX, pointer_obj=pointer, rotulo="pointer/blind_novo")
FINAL_NOVO = d3.resumir(FINAL_NOVO_LINHAS)
FINAL_V2REAL_LINHAS = avaliar_conjunto("pointer", BLIND_V2_REAL_EX, pointer_obj=pointer, rotulo="pointer/blind_v2_real")
FINAL_V2REAL = d3.resumir(FINAL_V2REAL_LINHAS)
FINAL_LEGADO = resumir_legado(avaliar_legado("pointer", BLIND_V2_LEGADO_EX, pointer_obj=pointer, rotulo="pointer/blind_v2_legado"))
PRESERVACAO = preservacao_next_token(pointer)

def tabela(base, final, rotulo):
    print(f"\n--- {rotulo}: base → pointer ---", flush=True)
    f = lambda v: "  —  " if v is None else f"{100*v:5.1f}%"
    for t in d3.TAREFAS:
        b, p = base[t], final[t]
        print(f"{t:12s} exact-name {f(b['exact_name'])} → {f(p['exact_name'])} | "
              f"exact-name+fim {f(b['exact_name_fim'])} → {f(p['exact_name_fim'])} | "
              f"semântica {f(b['semantica'])} → {f(p['semantica'])} | "
              f"nome duplicado {f(p['nome_duplicado'])}", flush=True)
    print(f"semântica geral {f(base['semantica_geral'])} → {f(final['semantica_geral'])}", flush=True)

tabela(BASE_NOVO, FINAL_NOVO, "blind novo (formato real)")
tabela(BASE_V2REAL, FINAL_V2REAL, "blind v2 em formato real (regressão)")
print("\nblind v2 legado — base:", json.dumps(BASE_LEGADO, ensure_ascii=False), flush=True)
print("blind v2 legado — pointer v3:", json.dumps(FINAL_LEGADO, ensure_ascii=False), flush=True)
print("PRESERVAÇÃO EXTRA:", json.dumps(PRESERVACAO, ensure_ascii=False), flush=True)

HASH_APOS_AVALIACAO = fingerprint_modulo(modelo)
if HASH_APOS_AVALIACAO != HASH_ANTES:
    raise SystemExit("ERRO: hash da base mudou durante avaliação.")


In [ ]:
# Célula 11 — gates congelados (blind novo) + salvamento em pasta NOVA
print("PÓS-CHECK — gates e salvamento", flush=True)

HASH_DEPOIS = fingerprint_modulo(modelo)
BASE_UNCHANGED = HASH_DEPOIS == HASH_ANTES
if not BASE_UNCHANGED:
    raise SystemExit("ERRO: base mudou. Nada será salvo como resultado válido.")

FN = FINAL_NOVO
pos_vals = [FN[f"posicao_{p}"] for p in d3.POSICOES]
GATES = {
    "exact_name_fim_copia>=0.70": FN["exact_name_fim_copia"] >= 0.70,
    "exact_name_distant>=0.65": FN["lexical_distant"] >= 0.65,
    "abs(close-distant)<=0.10": abs(FN["lexical_close"] - FN["lexical_distant"]) <= 0.10,
    "max(posicao)-min(posicao)<=0.10": max(pos_vals) - min(pos_vals) <= 0.10,
    "regressao_funcao<=5pp": BASE_NOVO["funcao"]["semantica"] - FN["funcao"]["semantica"] <= 0.05 + 1e-9,
    "regressao_semantica_geral<=2pp": BASE_NOVO["semantica_geral"] - FN["semantica_geral"] <= 0.02 + 1e-9,
    "transformer_hash_igual": BASE_UNCHANGED,
}
GATE_PASS = all(GATES.values())
EXTRA_CHECKS = {
    "preservation_top1>=0.90": PRESERVACAO["top1_agreement"] >= 0.90,
    "preservation_mean_TV<=0.15": PRESERVACAO["mean_total_variation"] <= 0.15,
}

def por_tarefa(r):
    return {t: {k: r[t][k] for k in ("exact_name", "exact_name_fim", "semantica", "resposta_exata", "nome_duplicado")}
            for t in d3.TAREFAS}

resultado = {
    "protocol": "pointer_copy_causal_v3",
    "created_at": datetime.now().isoformat(),
    "repo_branch": REPO_BRANCH,
    "repo_commit": commit,
    "dados_congelados_sha256": ESPERADO_SHA,
    "base_checkpoint": str(BASE_RUN),
    "base_phase": "05_05_dialogo_estado",
    "base_file_sha256": SHA_PESOS,
    "tokenizer_sha256": SHA_TOKENIZER,
    "base_hash_before": HASH_ANTES,
    "base_hash_after": HASH_DEPOIS,
    "base_unchanged": BASE_UNCHANGED,
    "base_parameters": int(N_BASE),
    "pointer_parameters": int(N_POINTER),
    "pointer_rank": int(RANK),
    "pointer_reinicializado": True,
    "seed": int(SEED),
    "train_steps_requested": int(PASSOS),
    "train_steps_done": int(passos_feitos),
    "melhor_passo_dev": melhor_passo,
    "train_history": historico,
    "copiabilidade_blind_novo": COPIABILIDADE,
    "teto_teacher_forcing_blind_novo": TETO,
    "blind_novo": {"baseline": BASE_NOVO, "final": FINAL_NOVO,
                   "por_tarefa_baseline": por_tarefa(BASE_NOVO), "por_tarefa_final": por_tarefa(FINAL_NOVO)},
    "blind_v2_formato_real": {"baseline": BASE_V2REAL, "final": FINAL_V2REAL},
    "blind_v2_legado": {"baseline": BASE_LEGADO, "final": FINAL_LEGADO},
    "preservation_extra": PRESERVACAO,
    "gates": GATES,
    "extra_checks": EXTRA_CHECKS,
    "gate_pass": bool(GATE_PASS),
    "porta_conversacional": "pendente — exigida mesmo se gate_pass=true",
    "ativo_no_chat": False,
    "promovido": False,
    "linhas_blind_novo_baseline": BASE_NOVO_LINHAS,
    "linhas_blind_novo_final": FINAL_NOVO_LINHAS,
}

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
DEST_ROOT = DRIVE_CHECKPOINTS / "experimento_pointer_copy_v3"
DEST_ROOT.mkdir(parents=True, exist_ok=True)
DEST = DEST_ROOT / f"pointer_copy_v3_{stamp}"
if "experimento_pointer_copy/" in str(DEST) + "/":
    raise SystemExit("ERRO: destino apontaria para a pasta do v2")
DEST.mkdir(parents=False, exist_ok=False)

torch.save(
    {
        "state_dict": {k: v.detach().cpu() for k, v in pointer.state_dict().items()},
        "protocol": "pointer_copy_causal_v3",
        "rank": RANK, "dimensao": cfg.dimensao, "vocabulario": cfg.vocabulario,
        "base_checkpoint": str(BASE_RUN), "base_hash": HASH_DEPOIS,
        "base_file_sha256": SHA_PESOS, "tokenizer_sha256": SHA_TOKENIZER,
        "gate_pass": bool(GATE_PASS), "ativo_no_chat": False, "promovido": False,
    },
    DEST / "pointer_head_v3.pt",
)
(DEST / "resultado_pointer_copy_v3.json").write_text(json.dumps(resultado, ensure_ascii=False, indent=2), encoding="utf-8")
config_exp = {
    "protocol": "pointer_copy_causal_v3", "seed": SEED, "rank": RANK, "p_gen_init": 0.90,
    "batch": BATCH, "no_copy_per_batch": N_NO_COPY, "lr": LR, "steps_requested": PASSOS,
    "steps_done": passos_feitos, "loss": "mixture_ce + 0.50*entity_copy_ce + 0.10*gate_bce + 0.50*fim_copia_ce",
    "source_copy_policy": "prompt_original_only_excluding_specials",
    "prompt_format": "lp.fonte_dialogo(texto completo)", "resposta_format": "lp.codificar_texto(texto) + <fim>",
    "blinds_congelados_antes_do_treino": True, "ativo_no_chat": False, "promovido": False,
}
(DEST / "config_experimento_v3.json").write_text(json.dumps(config_exp, ensure_ascii=False, indent=2), encoding="utf-8")

def pct(v):
    return "—" if v is None else f"{100*v:.1f}%"

linhas_tab = "\n".join(
    f"{t:12s} exact-name {pct(BASE_NOVO[t]['exact_name'])} -> {pct(FN[t]['exact_name'])} | "
    f"exact-name+fim {pct(BASE_NOVO[t]['exact_name_fim'])} -> {pct(FN[t]['exact_name_fim'])} | "
    f"semantica {pct(BASE_NOVO[t]['semantica'])} -> {pct(FN[t]['semantica'])}"
    for t in d3.TAREFAS)
readme = f"""CRIVO — pointer_copy_causal_v3
base: {BASE_RUN}
repo: {REPO_BRANCH} @ {commit}
base_file_sha256: {SHA_PESOS}
base_hash_before/after: {HASH_ANTES} / {HASH_DEPOIS}
pointer_params: {N_POINTER} (reinicializado; mesma arquitetura do v2)
steps_done: {passos_feitos} | melhor passo dev: {melhor_passo}

teto teacher forcing exact-name+fim (cópia pura): {pct(TETO['copia']['teto_exact_name_fim'])}

blind novo (formato real), base -> pointer:
{linhas_tab}
semantica_geral {pct(BASE_NOVO['semantica_geral'])} -> {pct(FN['semantica_geral'])}

blind v2 legado pure_copy_exact: base {pct(BASE_LEGADO['pure_copy_exact_match'])} -> v3 {pct(FINAL_LEGADO['pure_copy_exact_match'])}

gates:
{json.dumps(GATES, ensure_ascii=False, indent=2)}
extra_checks:
{json.dumps(EXTRA_CHECKS, ensure_ascii=False, indent=2)}

gate_pass: {GATE_PASS}
porta_conversacional: pendente
ativo_no_chat: false
promovido: false
"""
(DEST / "README_resultado_v3.txt").write_text(readme, encoding="utf-8")

for p in (DEST / "pointer_head_v3.pt", DEST / "resultado_pointer_copy_v3.json",
          DEST / "config_experimento_v3.json", DEST / "README_resultado_v3.txt"):
    if not p.is_file() or p.stat().st_size == 0:
        raise SystemExit(f"ERRO de persistência: {p}")

print("\n================ RESULTADO v3 ================", flush=True)
print(readme, flush=True)
print("SALVO EM:", DEST, flush=True)
print(f"gate_pass={GATE_PASS} | porta conversacional pendente | ativo_no_chat=false | promovido=false", flush=True)
